# Parse IOT and SUT into `Dataset`

This tutorial shows how to parse packaged Excel fixtures directly into the new
`mario.Dataset` model.


## Resolve the repository root

The notebook is stored under `doc/source/tutorials`, so we locate the repository
root dynamically before opening the test workbooks.


In [ ]:
from pathlib import Path

import mario


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "mario" / "test" / "IOT.xlsx").exists():
            return candidate
    raise FileNotFoundError("Could not locate the MARIO repository root.")


REPO_ROOT = find_repo_root(Path.cwd())
IOT_PATH = REPO_ROOT / "mario" / "test" / "IOT.xlsx"
SUT_PATH = REPO_ROOT / "mario" / "test" / "SUT.xlsx"

print(IOT_PATH)
print(SUT_PATH)


## Parse an IOT dataset


In [ ]:
iot = mario.parse_dataset_from_excel(
    str(IOT_PATH),
    table="IOT",
    mode="flows",
    name="Tutorial IOT",
)

print(iot.table_kind.value)
print(iot.list_blocks())


## Parse a SUT dataset

For SUT, the parser stores split-native canonical blocks such as `U`, `S`, `Xa`,
`Xc`, `Ya`, `Yc`, `Va`, `Vc`, `Ea` and `Ec`. Unified blocks such as `Z`, `Y`, `V`,
`E` and `X` are reconstructed on demand by the compute layer.


In [ ]:
sut = mario.parse_dataset(
    "excel",
    path=str(SUT_PATH),
    table="SUT",
    mode="flows",
    name="Tutorial SUT",
)

print(sut.table_kind.value)
print(sut.list_blocks())


In [ ]:
Z = sut.compute("Z")
Y = sut.compute("Y")
V = sut.compute("V")
E = sut.compute("E")
X = sut.compute("X")

print(Z.shape)
print(Y.shape)
print(V.shape)
print(E.shape)
print(X.shape)


In [ ]:
print(sut.get_index("a")[:3])
print(sut.get_index("c")[:3])


## Compare with the Database fixture

The domain semantics remain aligned with the historical package. For example, the
unified `Z` reconstructed from the SUT `Dataset` matches the packaged SUT database.


In [ ]:
sut_db = mario.load_test("SUT")
dataset_z = sut.compute("Z")

(dataset_z == sut_db.Z).all().all()


This is the main parser story in the restructured package: keep MARIO's table
grammar, but parse into a smaller and more explicit internal model.
